#### Imports

In [0]:
import sys
from pathlib import Path

DATABRICKS_ROOT = str(Path.cwd().parent)

if DATABRICKS_ROOT not in sys.path:
    sys.path.insert(0, DATABRICKS_ROOT)

from config.datasets import get_dataset_config
from utilities.io_utils import read_dataset, write_delta_table

#### Workflow/Notebook parameter

In [0]:
dbutils.widgets.text(
    "dataset_name",
    "customers",
    "Dataset Name"
)
dataset_name = dbutils.widgets.get("dataset_name").strip()


#### Dataset-specific configuration

In [0]:
config = get_dataset_config(dataset_name)

print(f"Processing dataset: {dataset_name}")
print(f"Source path: {config['landing_path']}")
print(f"Target table: {config['bronze_table']}")
print(f"Primary key: {config['primary_key']}")

#### Reading landing data

In [0]:
raw_df = read_dataset(
    spark=spark,
    config=config
)
raw_df.printSchema()

#### Write to Bronze

In [0]:
write_delta_table(
    spark=spark,
    dataframe=raw_df,
    table_name=config["bronze_table"]
)

#### Validating the resulting table

In [0]:
write_delta_table(
    spark=spark,
    dataframe=raw_df,
    table_name=config["bronze_table"]
)

bronze_df = spark.table(config["bronze_table"])

print(f"Dataset: {dataset_name}")
print(f"Rows written: {bronze_df.count():,}")
print(f"Target table: {config['bronze_table']}")

display(bronze_df.limit(10))